<a href="https://colab.research.google.com/github/YuriyTabinskuy/-_-_5_-.ipynb/blob/main/%D0%95%D0%BA%D0%B71.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install deap

import random
import multiprocessing
from deap import base, creator, tools, algorithms

def evaluate_sphere(individual):
    return sum(x**2 for x in individual),

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_float", random.uniform, -5.12, 5.12)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=10)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate_sphere)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)


def run_island(island_id, pop_size, generations, pipe):
    random.seed()
    pop = toolbox.population(n=pop_size)

    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)

    for gen in range(1, generations + 1):
        pop = algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=1, verbose=False)[0]


        if gen % 10 == 0:
            best_ind = tools.selBest(pop, 1)[0]
            pipe.send((island_id, gen, best_ind, best_ind.fitness.values[0]))

    return tools.selBest(pop, 1)[0]


if __name__ == "__main__":
    NUM_ISLANDS = 4
    POP_SIZE_PER_ISLAND = 50
    GENERATIONS = 50

    pipes = [multiprocessing.Pipe() for _ in range(NUM_ISLANDS)]
    processes = []

    print(style := "-"*50)
    print(f"Запуск паралельної острівної моделі ГА ({NUM_ISLANDS} островів)...")
    print(style)


    for i in range(NUM_ISLANDS):
        p = multiprocessing.Process(target=run_island, args=(i, POP_SIZE_PER_ISLAND, GENERATIONS, pipes[i][1]))
        processes.append(p)
        p.start()


    for i in range(NUM_ISLANDS):
        if pipes[i][0].poll(timeout=5):
            island_id, gen, ind, fit = pipes[i][0].recv()
            print(style := f"Острів {island_id} | Покоління {gen} | Найкраща пристосованість: {fit:.4f}")

    for p in processes:
        p.join()

    print("-" * 50)
    print("Паралельні обчислення завершено успішно.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.9/866.9 kB 20.3 MB/s eta 0:00:00
--------------------------------------------------
Запуск паралельної острівної моделі ГА (4 островів)...
--------------------------------------------------
Острів 0 | Покоління 10 | Найкраща пристосованість: 4.5170
Острів 1 | Покоління 10 | Найкраща пристосованість: 16.4334
Острів 2 | Покоління 10 | Найкраща пристосованість: 7.4992
Острів 3 | Покоління 10 | Найкраща пристосованість: 3.1875
--------------------------------------------------
Паралельні обчислення завершено успішно.
